# Measure time-evolved expectation values in `qdk-chemistry`

## This notebook demonstrates the `EvolveAndMeasure` algorithm for Hamiltonian time evolution and observable measurement using `qdk-chemistry`. It shows how to:

1. Define a qubit Hamiltonian with time-dependent parameters
2. Build a Trotter evolution circuit
3. Run the simulation **without noise** using the QDK full-state simulator
4. Run the simulation **with noise** using both the QDK and Qiskit Aer backends
5. Optionally transpile to a target basis gate set before execution

In [ ]:
import numpy as np

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import AlgorithmRef, LatticeGraph, QuantumErrorProfile, QubitHamiltonian
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)

We define two qubit Hamiltonians representing alternating time-evolution steps (e.g., a driven or Floquet-like protocol). The observable is `ZZ`, measured after the full evolution sequence.

The lists `hamiltonians` and `time_steps` define the discretized time-dependent Hamiltonian $H(t)$ as:

$H(t_i) = H_i$, where

$H_i \in $ `hamiltonians` = $\left[H_1,\dots,H_n\right]$,

$t_i \in $`time_steps` = $\left[t_1,\dots,t_n\right]$

In [ ]:
lattice = LatticeGraph.chain(2)

hamiltonian_p = create_ising_hamiltonian(lattice, j=1.0, h=0.5)
hamiltonian_m = create_ising_hamiltonian(lattice, j=1.0, h=-0.5)

steps = 20
hamiltonians = [hamiltonian_p, hamiltonian_m] * (steps // 2)
time_steps = [float((t + 1) / 10) for t in range(steps)]

observable = QubitHamiltonian(["ZZ"], np.array([1.0]))

## Exact evolution via matrix exponential

Convert the Hamiltonians to sparse matrices and compute the exact time evolution $|\psi(t)\rangle = \prod_i e^{-i H_i \Delta t_i} |\psi_0\rangle$.

In [ ]:
from scipy.sparse.linalg import expm_multiply

# Convert Hamiltonians to sparse matrices
H_p = hamiltonian_p.to_matrix(sparse=True)
H_m = hamiltonian_m.to_matrix(sparse=True)

num_qubits = H_p.shape[0].bit_length() - 1

# Initial state |00...0⟩
psi = np.zeros(2**num_qubits, dtype=complex)
psi[0] = 1.0

# Time-evolve: apply exp(-i H_i dt_i) for each step
dt_list = [time_steps[0]] + [time_steps[i] - time_steps[i - 1] for i in range(1, len(time_steps))]

for ham, dt in zip(hamiltonians, dt_list):
    H_sparse = H_p if ham is hamiltonian_p else H_m
    psi = expm_multiply(-1j * H_sparse * dt, psi)

# Compute exact ⟨ZZ⟩
Z = np.array([1, -1], dtype=complex)
ZZ_diag = np.kron(Z, Z)
zz_exact = np.real(np.conj(psi) @ (ZZ_diag * psi))

print(f"Exact ⟨ZZ⟩: {zz_exact:.6f}")

## Noiseless simulation (QDK full-state simulator)
Set up the `measure_simulation` algorithm and configure its nested algorithms via settings.

In [ ]:
measure_simulation = create("measure_simulation", "evolve_and_measure")
measure_simulation.settings().set(
    "evolution_builder",
    AlgorithmRef("hamiltonian_unitary_builder", "trotter", num_divisions=2, order=1, optimize_term_ordering=True),
)

Run the evolution and measure `⟨ZZ⟩` without any noise using the QDK full-state simulator.

In [ ]:
measure_simulation.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qdk_full_state_simulator"),
)

measurements_noiseless = measure_simulation.run(
    hamiltonians,
    times=time_steps,
    observables=[observable],
    shots=10000,
)

zz_noiseless = measurements_noiseless[0][0].energy_expectation_value
print(f"Noiseless ⟨ZZ⟩: {zz_noiseless:.6f}")

## Define a noise profile

`QuantumErrorProfile` provides a backend-agnostic way to specify depolarizing noise on individual gates. This profile can be used with both the QDK and Qiskit Aer simulators.

> **Note:** The QDK full-state simulator applies noise at the *native gate level* of the compiled QIR. If the circuit uses high-level Pauli exponentials (e.g. via `PauliSequenceMapper`), those are executed natively without decomposition into `cx`/`rz` gates — so noise on `cx` won't apply. To see noise with the QDK simulator, either use `basis_gates` to force decomposition, or define noise on the native gates (`rzz`, `rx`, etc.).

In [ ]:
qdk_error_profile = QuantumErrorProfile(
    name="demo_noise",
    description="Light depolarizing noise on common gates",
    errors={
        "cx":  {"type": "depolarizing_error", "rate": 0.01,  "num_qubits": 2},
        "cz":  {"type": "depolarizing_error", "rate": 0.01,  "num_qubits": 2},
        "rzz": {"type": "depolarizing_error", "rate": 0.01,  "num_qubits": 2},
        "h":   {"type": "depolarizing_error", "rate": 0.001, "num_qubits": 1},
        "rz":  {"type": "depolarizing_error", "rate": 0.001, "num_qubits": 1},
        "rx":  {"type": "depolarizing_error", "rate": 0.001, "num_qubits": 1},
        "s":   {"type": "depolarizing_error", "rate": 0.001, "num_qubits": 1},
        "sdg": {"type": "depolarizing_error", "rate": 0.001, "num_qubits": 1},
    },
)

## Noisy simulation — QDK full-state simulator

Because `PauliSequenceMapper` produces native Pauli-exponential operations, and our noise profile includes `rzz` (the native two-qubit gate), the QDK simulator **will** apply noise to those operations.

In [ ]:
measurements_noisy_qdk = measure_simulation.run(
    hamiltonians,
    times=time_steps,
    observables=[observable],
    shots=10000,
    noise=qdk_error_profile,
)

zz_noisy_qdk = measurements_noisy_qdk[0][0].energy_expectation_value
print(f"Noisy QDK ⟨ZZ⟩: {zz_noisy_qdk:.6f}")

## Noisy simulation — Qiskit Aer simulator

The Qiskit Aer backend transpiles the circuit to primitive gates (`cx`, `rz`, `h`, …) before execution, so noise on those gates fires naturally.

In [ ]:
measure_simulation.settings().set(
    "circuit_executor",
    AlgorithmRef("circuit_executor", "qiskit_aer_simulator"),
)

measurements_noisy_aer = measure_simulation.run(
    hamiltonians,
    times=time_steps,
    observables=[observable],
    shots=10000,
    noise=qdk_error_profile,
)

zz_noisy_aer = measurements_noisy_aer[0][0].energy_expectation_value
print(f"Noisy Aer ⟨ZZ⟩: {zz_noisy_aer:.6f}")

## Compare results

Side-by-side comparison of the noiseless and noisy expectation values.

In [ ]:
print("Simulator               ⟨ZZ⟩")
print("─" * 36)
print(f"Exact                 {zz_exact: .6f}")
print(f"QDK  (noiseless)      {zz_noiseless: .6f}")
print(f"QDK  (noisy)          {zz_noisy_qdk: .6f}")
print(f"Aer  (noisy)          {zz_noisy_aer: .6f}")